In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import datetime
from selenium import webdriver
from time import sleep
import os
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'GB TCIFSC' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running GB TCIFSC Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------


#------------------------------------------------ Begin_Variable ----------------------------------------
regdict={
        regulatorName+' 1': 'https://www.tcifsc.tc/banks-licensed-banks',
        regulatorName+' 2': 'https://www.tcifsc.tc/company-managers-company-managers-agents',
        regulatorName+' 3': 'https://www.tcifsc.tc/international-insurance',
        regulatorName+' 4': 'https://www.tcifsc.tc/domestic-insurance',
        regulatorName+' 5': 'https://www.tcifsc.tc/trust-companies-licensed-trust-companies',
        regulatorName+' 6': 'https://www.tcifsc.tc/money-transmitters-licensed-money-transmitters',
        regulatorName+' 7 1': 'https://www.tcifsc.tc/investments-mutual-funds-mutual-fund-administrators',
        regulatorName+' 7 2': 'https://www.tcifsc.tc/investments-mutual-funds-exempt-mutual-funds',                   
        regulatorName+' 8': 'https://www.tcifsc.tc/investment-dealers-investment-dealers-and-advisors',        
        }

Typology={
        regulatorName+' 1': 'Licensed Banks',
        regulatorName+' 2': 'Company Managers',
        regulatorName+' 3': 'International Insurance',
        regulatorName+' 4': 'Domestic Insurance',
        regulatorName+' 5': 'Licensed Trust Companies',
        regulatorName+' 6': 'Licensed Money Transmitters',
        regulatorName+' 7 1': 'Mutual Funds',
        regulatorName+' 7 2': 'Mutual Funds',                   
        regulatorName+' 8': 'Investment Dealers',        
        }


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')

In [ ]:
for index, reg in enumerate(regdict):
    print(f"[INFO] : Working {index+1}/{len(regdict)} _({reg})_ ")
    response = requests.get(regdict[reg],verify=False)
    soup = BeautifulSoup(response.text, 'html.parser')

    if reg == 'GB TCIFSC 1' or  reg == 'GB TCIFSC 2':
        # Find all tables
        tables = soup.find_all('table')

        # Process each table
        for idx, table in enumerate(tables, start=1):
            #print(f"\n--- Table {idx} ---")
            
            # Extract headers
            headers = [th.get_text(strip=True) for th in table.find_all('th')]
            
            # Extract rows
            for tr in table.find_all('tr')[1:]:  # skip header row
                cells = [td.get_text(" ", strip=True) for td in tr.find_all('td')]
                if cells:
                    row_data = dict(zip(headers, cells))
                    
                    # Split second column into Address, Telephone, Fax
                    if len(cells) > 1:
                        contact_info = cells[1]
                        
                        # Regex patterns for Telephone and Fax
                        tel_match = re.search(r'Tele:\s*([\+\d\s\-]+)', contact_info)
                        fax_match = re.search(r'Fax:\s*([\+\d\s\-]+)', contact_info)
                        
                        telephone = tel_match.group(1).strip() if tel_match else ''
                        fax = fax_match.group(1).strip() if fax_match else ''
                        
                        # Remove Tel and Fax from address
                        address = re.sub(r'Tele:.*|Fax:.*', '', contact_info).strip()
                        
                        # Add new variables
                        row_data['Address'] = address
                        row_data['Telephone'] = telephone
                        row_data['Fax'] = fax
                    
                    #print(row_data)
                    if reg == 'GB TCIFSC 1':
                        name = row_data['Name']
                        address_ = row_data['Address']
                        website_ = row_data['Website']
                        tele_ = row_data['Telephone']
                        fax_ = row_data['Fax']
                        
                    elif reg == 'GB TCIFSC 2':
                        name = row_data['NAME OF COMPANY']
                        class_of_licence = row_data['CLASS OF LICENCE']
                        address_ = row_data['ADDRESS']
                        website_ = row_data['WEBSITE ADDRESS']
                    sqldict['Name'].append(name)
                    sqldict['Address_1'].append(address_)
                    sqldict['Website'].append(website_)
                    if reg == 'GB TCIFSC 1':
                        sqldict['Phone'].append(tele_)
                        sqldict['Fax'].append(fax_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')

                    sqldict = bourange_same_length_array(sqldict)
    elif reg == 'GB TCIFSC 5' or reg == 'GB TCIFSC 7 1'or reg == 'GB TCIFSC 7 2' or reg == 'GB TCIFSC 8':
            container = soup.find('div',class_='container-box container cms')
            row_div = container.find('div', class_='row')
            url_div = row_div.find_all('div')[-1]
            ul_tags = url_div.find_all('ul')
            if ul_tags:
                #print("Found UL lists. Processing as list layout...")
                for ul in ul_tags:
                    for li in ul.find_all('li'):
                        text = li.get_text(strip=True)
                        name = text
                        #print({"Item": text})
                        sqldict['Name'].append(name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
            else:
                print("No tables or UL lists found in <div class='row'>.")  
    elif reg == 'GB TCIFSC 3' or reg == 'GB TCIFSC 4' :
        embed_divs = soup.find_all('div', class_='raw-html-embed')
        links = []

        for div in embed_divs:
            for li in div.find_all('li'):
                text = li.get_text(strip=True)
                a_tag = li.find('a')
                href = a_tag['href'] if a_tag else None
                print(f"Item: {text}, Link: {href}")
                if href:
                    links.append(href)

        for link in links:
            print(f"\n--- Visiting: {link} ---")
            if not link.startswith('http'):
                link = 'https://www.tcifsc.tc' + link  # handle relative URLs
            
            sub_response = requests.get(link,verify=False)
            sub_soup = BeautifulSoup(sub_response.text, 'html.parser')
            container = sub_soup.find('div',class_='container-box container cms')
            row_div = container.find('div', class_='row')
            # Check if it contains tables
            tables = row_div.find_all('table')
            if tables:
                #print("Found tables. Processing as table layout...")
                for idx, table in enumerate(tables, start=1):
                    #print(f"\n--- Table {idx} ---")
                    # Extract headers
                    headers = [th.get_text(strip=True) for th in table.find_all('th')]
                    
                    # Extract rows
                    for tr in table.find_all('tr')[1:]:  # skip header row
                        cells = [td.get_text(" ", strip=True) for td in tr.find_all('td')]
                        if cells:
                            row_data = dict(zip(headers, cells))
                            
                            # Split second column into Address, Telephone, Fax
                            if len(cells) > 1:
                                contact_info = cells[1]
                                
                                tel_match = re.search(r'Tele:\s*([\+\d\s\-]+)', contact_info)
                                fax_match = re.search(r'Fax:\s*([\+\d\s\-]+)', contact_info)
                                
                                telephone = tel_match.group(1).strip() if tel_match else ''
                                fax = fax_match.group(1).strip() if fax_match else ''
                                
                                address = re.sub(r'Tele:.*|Fax:.*', '', contact_info).strip()
                                
                                row_data['Address'] = address
                                row_data['Telephone'] = telephone
                                row_data['Fax'] = fax
                            
                            #print(row_data)
                            name = row_data['NAME']
                            address_ = row_data['PHYSICAL ADDRESS']
                            website_ = row_data['WEBSITE ADDRESS']
                            sqldict['Name'].append(name)
                            sqldict['Address_1'].append(address_)
                            sqldict['Website'].append(website_)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['RegCtry'].append(reg.split()[0])
                            sqldict['RegCode'].append(reg.split()[1])
                            sqldict['ListCode'].append(reg.split()[2])
                            sqldict['ListName'].append(Typology[reg])
                            sqldict['RegulationType'].append('Regulated')

                    sqldict = bourange_same_length_array(sqldict)
            else:
                # If no tables, check for UL lists
                url_div = row_div.find_all('div')[-1]
                ul_tags = url_div.find_all('ul')
                if ul_tags:
                    #print("Found UL lists. Processing as list layout...")
                    for ul in ul_tags:
                        for li in ul.find_all('li'):
                            text = li.get_text(strip=True)
                            name = text
                            # print({"Item": text})
                            sqldict['Name'].append(name)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['RegCtry'].append(reg.split()[0])
                            sqldict['RegCode'].append(reg.split()[1])
                            sqldict['ListCode'].append(reg.split()[2])
                            sqldict['ListName'].append(Typology[reg])
                            sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)
                else:
                    print("No tables or UL lists found in <div class='row'>.")   
    elif reg == 'GB TCIFSC 6':
        container = soup.find('div',class_='container-box container cms')
        row_div = container.find('div', class_='row')
        cards = row_div.find_all('div', class_='card')

        for card in cards:
            # Create a fresh dictionary for each card
            data = {}

            # Extract company name
            name = card.find('h5').get_text(strip=True)
            data['CompanyName'] = name
            
            # Extract all <p> elements
            for p in card.find_all('p'):
                key = p.find('strong').get_text(strip=True).replace(':', '')
                a_tag = p.find('a')
                if a_tag:
                    value = a_tag.get_text(strip=True)
                else:
                    value = p.get_text(strip=True).replace(p.find('strong').get_text(strip=True), '').strip()
                data[key] = value

            # Print structured data for this card
            #print(data)

            # Safely access keys (use .get to avoid errors)
            name_ = data.get('CompanyName', '')
            address_ = data.get('Address', '')
            tele_ = data.get('Tel', '')
            fax_ = data.get('Fax', '')
            #print(f"Name: {name_}, Address: {address_}, Tel: {tele_}, Fax: {fax_}")
            sqldict['Name'].append(name_)
            sqldict['Address_1'].append(address_)
            sqldict['Phone'].append(tele_)
            sqldict['Fax'].append(fax_)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
sqldict = bourange_same_length_array(sqldict)
        

[INFO] : Working 1/9 _(GB TCIFSC 1)_ 
[INFO] : Working 2/9 _(GB TCIFSC 2)_ 
[INFO] : Working 3/9 _(GB TCIFSC 3)_ 
Item: Insurance Managers, Link: https://www.tcifsc.tc/international-insurance-insurance-managers
Item: Non US Direct Writers, Link: https://www.tcifsc.tc/international-insurance-non-us-direct-writers

--- Visiting: https://www.tcifsc.tc/international-insurance-insurance-managers ---

--- Visiting: https://www.tcifsc.tc/international-insurance-non-us-direct-writers ---
[INFO] : Working 4/9 _(GB TCIFSC 4)_ 
Item: Insurance Companies, Link: https://www.tcifsc.tc/domestic-insurance-insurance-companies
Item: Insurance Brokers, Link: https://www.tcifsc.tc/domestic-insurance-insurance-brokers
Item: Insurance Agents, Link: https://www.tcifsc.tc/domestic-insurance-insurance-agents
Item: Insurance Sub Agents, Link: https://www.tcifsc.tc/domestic-insurance-insurance-sub-agents

--- Visiting: https://www.tcifsc.tc/domestic-insurance-insurance-companies ---

--- Visiting: https://www.tc

In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename,  index=False)
driver.quit()
sleep(3)